# LeetCode 2161 — Partition Array According to Given Pivot bằng Python

Link đề: https://leetcode.com/problems/partition-array-according-to-given-pivot/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Simulation

Skeleton LeetCode:

```python
class Solution:
    def pivotArray(self, nums: List[int], pivot: int) -> List[int]:
        ...
```

Bài toán **phân hoạch ổn định (stable partition)** thành ba nhóm `< pivot`, `= pivot`, `> pivot`. Phân hoạch của Quicksort quen thuộc **không ổn định**, nên đây là bài tốt để hiểu khác biệt giữa "chia nhóm" và "chia nhóm **giữ thứ tự**". Notebook đi từ **đổi chỗ kề nhau kiểu bubble `O(n²)`** → **sắp xếp ổn định theo nhãn nhóm** → **ba danh sách** → **một mảng kết quả với hai con trỏ chạy ngược chiều** → **phân hoạch ổn định tại chỗ bằng chia để trị + xoay mảng**.

## 1. Mục tiêu học

1. Hiểu khái niệm **ổn định** (stable): các phần tử cùng nhóm giữ nguyên thứ tự tương đối như ban đầu.
2. Thấy vì sao phân hoạch Lomuto/Hoare của Quicksort **không** giải được bài này, bằng phản ví dụ chạy thật.
3. Cài đặt phân hoạch ổn định `O(n)` bằng ba danh sách, và bằng **một mảng kết quả + hai con trỏ** (điền nhóm nhỏ từ trái sang, nhóm lớn từ phải sang khi duyệt ngược).
4. Dùng tính ổn định của **Timsort** (`sorted` của Python) với khóa nhãn nhóm.
5. Biết kỹ thuật **phân hoạch ổn định tại chỗ** `O(n log n)` thời gian, `O(log n)` bộ nhớ bằng chia để trị + xoay đoạn.

## 2. Đề bài & ví dụ

Cho mảng `nums` (đánh số từ 0) và số `pivot` (chắc chắn xuất hiện trong `nums`). Sắp lại `nums` sao cho:

1. Mọi phần tử **nhỏ hơn** `pivot` đứng trước mọi phần tử **lớn hơn** `pivot`.
2. Mọi phần tử **bằng** `pivot` đứng **giữa** hai nhóm trên.
3. Thứ tự tương đối **trong nhóm nhỏ hơn** và **trong nhóm lớn hơn** được giữ nguyên: nếu `i < j` và cả hai cùng nhỏ hơn (hoặc cùng lớn hơn) `pivot` thì vị trí mới của `nums[i]` đứng trước vị trí mới của `nums[j]`.

Trả về mảng sau khi sắp lại.

**Ví dụ 1:** `nums = [9,12,5,10,14,3,10]`, `pivot = 10` → `[9,5,3,10,10,12,14]`.

**Ví dụ 2:** `nums = [-3,4,3,2]`, `pivot = 2` → `[-3,2,4,3]`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `−10⁶ ≤ nums[i] ≤ 10⁶`, `pivot` là một phần tử của `nums`.

**Nhận xét:** với ba điều kiện trên, đáp án là **duy nhất** (thứ tự trong mỗi nhóm bị cố định; nhóm bằng `pivot` gồm các giá trị giống hệt nhau).

In [1]:
import random
import time
from typing import List

examples = [([9, 12, 5, 10, 14, 3, 10], 10, [9, 5, 3, 10, 10, 12, 14]), ([-3, 4, 3, 2], 2, [-3, 2, 4, 3])]

def label(x, pivot):
    return "<" if x < pivot else ("=" if x == pivot else ">")

for nums, pivot, exp in examples:
    print(f"nums={nums}, pivot={pivot}")
    print("   nhãn :", [label(x, pivot) for x in nums])
    print("   kỳ vọng:", exp)

nums=[9, 12, 5, 10, 14, 3, 10], pivot=10
   nhãn : ['<', '>', '<', '=', '>', '<', '=']
   kỳ vọng: [9, 5, 3, 10, 10, 12, 14]
nums=[-3, 4, 3, 2], pivot=2
   nhãn : ['<', '>', '>', '=']
   kỳ vọng: [-3, 2, 4, 3]


## 3. Kiến thức nền

### 3.1 Sắp xếp ổn định

Một thuật toán sắp xếp **ổn định** nếu hai phần tử có **khóa bằng nhau** giữ nguyên thứ tự xuất hiện ban đầu. `sorted` và `list.sort` của Python (Timsort) **luôn ổn định**.

In [2]:
people = [("An", 3), ("Bình", 1), ("Chi", 3), ("Dũng", 1)]
print("sắp theo số, ổn định:", sorted(people, key=lambda p: p[1]))
print("-> 'Bình' vẫn trước 'Dũng', 'An' vẫn trước 'Chi'")

sắp theo số, ổn định: [('Bình', 1), ('Dũng', 1), ('An', 3), ('Chi', 3)]
-> 'Bình' vẫn trước 'Dũng', 'An' vẫn trước 'Chi'


### 3.2 Nhãn nhóm bằng phép so sánh

`(x > p) - (x < p)` cho `−1`, `0`, `1` tương ứng nhỏ hơn, bằng, lớn hơn (`True − False = 1`). Đây là "hàm dấu" của `x − p`.

In [3]:
p = 10
print([(x, (x > p) - (x < p)) for x in [9, 12, 10, 3]])

[(9, -1), (12, 1), (10, 0), (3, -1)]


### 3.3 List comprehension có điều kiện

In [4]:
nums = [9, 12, 5, 10, 14, 3, 10]
print("nhỏ hơn:", [x for x in nums if x < 10], "| bằng:", [x for x in nums if x == 10], "| lớn hơn:", [x for x in nums if x > 10])

nhỏ hơn: [9, 5, 3] | bằng: [10, 10] | lớn hơn: [12, 14]


### 3.4 Xoay đoạn mảng (rotate) bằng ba lần đảo ngược

Xoay đoạn `a[l:r]` sao cho phần `a[l:m]` và `a[m:r]` đổi chỗ (giữ thứ tự bên trong mỗi phần): đảo `a[l:m]`, đảo `a[m:r]`, rồi đảo cả `a[l:r]`. Dùng `O(1)` bộ nhớ phụ nếu đảo bằng hai con trỏ.

In [5]:
def reverse(a, l, r):          # đảo a[l:r] tại chỗ
    r -= 1
    while l < r:
        a[l], a[r] = a[r], a[l]; l += 1; r -= 1

def rotate(a, l, m, r):        # [A | B] -> [B | A] trên đoạn a[l:r], A = a[l:m]
    reverse(a, l, m); reverse(a, m, r); reverse(a, l, r)

a = [1, 2, 3, 4, 5, 6, 7]
rotate(a, 1, 3, 6)             # A = [2,3], B = [4,5,6]
print(a)

[1, 4, 5, 6, 2, 3, 7]


## 4. Ý tưởng nền tảng / trực giác

Gán cho mỗi phần tử một **nhãn**: `<`, `=`, `>`. Kết quả mong muốn là **mọi nhãn `<` trước, rồi `=`, rồi `>`**, và **trong cùng nhãn, giữ thứ tự gốc**.

Tính tay ví dụ 1: `nums = [9,12,5,10,14,3,10]`, `pivot = 10`.

| chỉ số | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| giá trị | 9 | 12 | 5 | 10 | 14 | 3 | 10 |
| nhãn | `<` | `>` | `<` | `=` | `>` | `<` | `=` |

Đọc lần lượt các phần tử nhãn `<`: `9, 5, 3`; nhãn `=`: `10, 10`; nhãn `>`: `12, 14`. Ghép lại: `[9,5,3,10,10,12,14]` ✓.

Đây chính là "**sắp xếp ổn định theo nhãn**" — chỉ có 3 giá trị nhãn nên làm được trong `O(n)` (giống đếm phân phối / counting sort ổn định).

In [6]:
nums, pivot = examples[0][0], examples[0][1]
for name, cond in [("<", lambda x: x < pivot), ("=", lambda x: x == pivot), (">", lambda x: x > pivot)]:
    picked = [(i, x) for i, x in enumerate(nums) if cond(x)]
    print(f"  nhóm {name}: (chỉ số, giá trị) = {picked}")

  nhóm <: (chỉ số, giá trị) = [(0, 9), (2, 5), (5, 3)]
  nhóm =: (chỉ số, giá trị) = [(3, 10), (6, 10)]
  nhóm >: (chỉ số, giá trị) = [(1, 12), (4, 14)]


## 5. Cách 1 — Brute Force / baseline: đổi chỗ kề nhau kiểu bubble sort theo nhãn

Lặp: nếu có hai phần tử **kề nhau** mà nhãn bên trái **lớn hơn** nhãn bên phải (ví dụ `>` đứng trước `<`), đổi chỗ chúng. Dừng khi không còn cặp như vậy. Vì chỉ đổi khi nhãn **khác nhau** (nghiêm ngặt lớn hơn), hai phần tử cùng nhãn **không bao giờ vượt qua nhau** ⇒ ổn định.

In [7]:
def pivot_bubble(nums, pivot):
    a = list(nums)
    key = lambda x: (x > pivot) - (x < pivot)
    changed = True
    while changed:
        changed = False
        for i in range(len(a) - 1):
            if key(a[i]) > key(a[i + 1]):        # chỉ đổi khi nhãn trái > nhãn phải (nghiêm ngặt)
                a[i], a[i + 1] = a[i + 1], a[i]
                changed = True
    return a

for nums_, p_, exp in examples:
    print(pivot_bubble(nums_, p_), "(kỳ vọng", exp, ")")

[9, 5, 3, 10, 10, 12, 14] (kỳ vọng [9, 5, 3, 10, 10, 12, 14] )
[-3, 2, 4, 3] (kỳ vọng [-3, 2, 4, 3] )


**Độ phức tạp:** `O(n²)` thời gian (số lần đổi chỗ = số **nghịch thế** theo nhãn, có thể tới `~n²/4`), `O(n)` bộ nhớ cho bản sao.

**Nút thắt:** mỗi lần chỉ dịch một bước. Nhưng ta đã biết **chính xác** mỗi phần tử phải đến đâu: vị trí = (số phần tử nhóm trước nó) + (số phần tử cùng nhóm đứng trước nó). ⇒ đặt thẳng vào chỗ, `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** kết quả = `[các x < pivot theo thứ tự gốc] + [pivot] * c + [các x > pivot theo thứ tự gốc]`, với `c` là số lần `pivot` xuất hiện.

**Quy tắc khi duyệt một lượt từ trái sang:**
- **nếu** `x < pivot` **thì** nối vào nhóm trái;
- **nếu** `x == pivot` **thì** tăng bộ đếm;
- **ngược lại** nối vào nhóm phải.

**Phiên bản một mảng kết quả:** đếm trước số phần tử nhỏ hơn `L` và bằng `E`. Khi đó nhóm trái chiếm `[0, L)`, nhóm giữa `[L, L+E)`, nhóm phải `[L+E, n)`. Duyệt **xuôi** để điền nhóm trái từ vị trí 0 tăng dần; duyệt **ngược** để điền nhóm phải từ vị trí `n−1` giảm dần (duyệt ngược + điền ngược = giữ thứ tự xuôi). Thậm chí làm được trong **một vòng lặp** với hai chỉ số `i` (xuôi) và `j = n−1−i` (ngược) — không cần đếm trước!

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp ổn định theo nhãn (Timsort)

In [8]:
def pivot_sorted(nums, pivot):
    return sorted(nums, key=lambda x: (x > pivot) - (x < pivot))

for nums_, p_, exp in examples:
    print(pivot_sorted(nums_, p_), "(kỳ vọng", exp, ")")

[9, 5, 3, 10, 10, 12, 14] (kỳ vọng [9, 5, 3, 10, 10, 12, 14] )
[-3, 2, 4, 3] (kỳ vọng [-3, 2, 4, 3] )


**Độ phức tạp:** `O(n log n)` trên lý thuyết; thực tế Timsort nhận ra các "run" và chạy nhanh. Ngắn gọn nhưng **dựa vào tính ổn định** của thư viện — trong C++ phải dùng `stable_sort`, không dùng `sort`.

### 7.2 Cách 3 — Ba danh sách (một lượt duyệt)

In [9]:
def pivot_three_lists(nums, pivot):
    less, greater = [], []
    equal = 0
    for x in nums:
        if x < pivot:
            less.append(x)
        elif x == pivot:
            equal += 1
        else:
            greater.append(x)
    return less + [pivot] * equal + greater

for nums_, p_, exp in examples:
    print(pivot_three_lists(nums_, p_), "(kỳ vọng", exp, ")")

[9, 5, 3, 10, 10, 12, 14] (kỳ vọng [9, 5, 3, 10, 10, 12, 14] )
[-3, 2, 4, 3] (kỳ vọng [-3, 2, 4, 3] )


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ phụ. Không cần lưu các phần tử bằng `pivot` — chúng giống hệt nhau, chỉ cần **đếm**.

### 7.3 Cách 4 — Một mảng kết quả, hai con trỏ chạy ngược chiều

Một vòng `for i in range(n)`:
- nhìn `nums[i]` (xuôi): nếu `< pivot` thì ghi vào `res[left]`, `left += 1`;
- nhìn `nums[n−1−i]` (ngược): nếu `> pivot` thì ghi vào `res[right]`, `right −= 1`.

Sau vòng lặp, đoạn `[left, right]` còn trống chính là chỗ của các `pivot`.

In [10]:
def pivot_two_pointers(nums, pivot):
    n = len(nums)
    res = [pivot] * n                  # điền sẵn pivot: đoạn giữa không cần ghi lại
    left, right = 0, n - 1
    for i in range(n):
        if nums[i] < pivot:
            res[left] = nums[i]; left += 1
        j = n - 1 - i
        if nums[j] > pivot:
            res[right] = nums[j]; right -= 1
    return res

for nums_, p_, exp in examples:
    print(pivot_two_pointers(nums_, p_), "(kỳ vọng", exp, ")")

[9, 5, 3, 10, 10, 12, 14] (kỳ vọng [9, 5, 3, 10, 10, 12, 14] )
[-3, 2, 4, 3] (kỳ vọng [-3, 2, 4, 3] )


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ phụ ngoài mảng kết quả. Mẹo khởi tạo `res = [pivot] * n` giúp không phải điền đoạn giữa.

**Vì sao hai con trỏ không đè lên nhau?** `left` cuối cùng bằng `L` (số phần tử nhỏ hơn), `right` cuối cùng bằng `n − 1 − G` (G = số phần tử lớn hơn); vì `L + G ≤ n − 1` (có ít nhất một `pivot`) nên các vùng ghi `[0, L)` và `(n−1−G, n−1]` rời nhau ở **mọi thời điểm**.

### 7.4 Cách 5 — Phân hoạch ổn định tại chỗ: chia để trị + xoay

Muốn sắp lại **ngay trên `nums`** với bộ nhớ phụ nhỏ mà vẫn ổn định (bài toán kinh điển `std::stable_partition` khi không cấp được bộ đệm). Ý tưởng:
1. Chia đôi `[l, r)` tại `m`; đệ quy phân hoạch ổn định mỗi nửa theo **vị từ** `P(x)` ("thuộc nhóm trái").
2. Mỗi nửa giờ có dạng `[P-đúng | P-sai]`. Toàn đoạn: `[T₁ | F₁ | T₂ | F₂]`. **Xoay** `[F₁ | T₂]` thành `[T₂ | F₁]` ⇒ `[T₁ T₂ | F₁ F₂]` — ổn định!
3. Làm hai lượt: lượt 1 với `P(x) = x < pivot` (tách nhóm trái khỏi phần còn lại), lượt 2 trên phần còn lại với `P(x) = x == pivot`.

Xoay bằng 3 lần đảo (mục 3.4), mỗi tầng đệ quy tốn `O(n)` ⇒ tổng `O(n log n)`, độ sâu đệ quy `O(log n)` (≈ 17 với `n = 10⁵`, an toàn).

In [11]:
def stable_partition(a, l, r, pred):
    """Phân hoạch ổn định a[l:r] theo pred; trả về vị trí ranh giới (số phần tử pred đúng + l)."""
    if r - l == 0:
        return l
    if r - l == 1:
        return l + 1 if pred(a[l]) else l
    m = (l + r) // 2
    b1 = stable_partition(a, l, m, pred)      # a[l:b1] đúng, a[b1:m] sai
    b2 = stable_partition(a, m, r, pred)      # a[m:b2] đúng, a[b2:r] sai
    rotate(a, b1, m, b2)                       # [F1 | T2] -> [T2 | F1]
    return b1 + (b2 - m)

def pivot_inplace(nums, pivot):
    a = list(nums)                              # sao chép để không phá đầu vào của người gọi (demo)
    b = stable_partition(a, 0, len(a), lambda x: x < pivot)
    stable_partition(a, b, len(a), lambda x: x == pivot)
    return a

for nums_, p_, exp in examples:
    print(pivot_inplace(nums_, p_), "(kỳ vọng", exp, ")")

[9, 5, 3, 10, 10, 12, 14] (kỳ vọng [9, 5, 3, 10, 10, 12, 14] )
[-3, 2, 4, 3] (kỳ vọng [-3, 2, 4, 3] )


**Độ phức tạp:** `O(n log n)` thời gian, `O(log n)` bộ nhớ (ngăn xếp đệ quy). Chậm hơn cách 3/4 nhưng là kỹ thuật đáng biết khi **bộ nhớ rất hạn chế**.

## 8. TRACE từng bước

### 8.1 Trace cách 4 trên ví dụ 1

In [12]:
def trace_two_pointers(nums, pivot):
    n = len(nums)
    res = ["_"] * n
    left, right = 0, n - 1
    print(f"  {'i':>2} | {'nums[i]':>7} {'nums[n-1-i]':>11} | {'left':>4} {'right':>5} | res")
    for i in range(n):
        j = n - 1 - i
        if nums[i] < pivot:
            res[left] = nums[i]; left += 1
        if nums[j] > pivot:
            res[right] = nums[j]; right -= 1
        print(f"  {i:>2} | {nums[i]:>7} {nums[j]:>11} | {left:>4} {right:>5} | {res}")
    for k in range(left, right + 1):
        res[k] = pivot
    print("  điền pivot vào đoạn", [left, right], "->", res)
    return res

_ = trace_two_pointers([9, 12, 5, 10, 14, 3, 10], 10)

   i | nums[i] nums[n-1-i] | left right | res
   0 |       9          10 |    1     6 | [9, '_', '_', '_', '_', '_', '_']
   1 |      12           3 |    1     6 | [9, '_', '_', '_', '_', '_', '_']
   2 |       5          14 |    2     5 | [9, 5, '_', '_', '_', '_', 14]
   3 |      10          10 |    2     5 | [9, 5, '_', '_', '_', '_', 14]
   4 |      14           5 |    2     5 | [9, 5, '_', '_', '_', '_', 14]
   5 |       3          12 |    3     4 | [9, 5, 3, '_', '_', 12, 14]
   6 |      10           9 |    3     4 | [9, 5, 3, '_', '_', 12, 14]
  điền pivot vào đoạn [3, 4] -> [9, 5, 3, 10, 10, 12, 14]


Quan sát: nhóm lớn hơn được điền **từ phải sang**: `14` vào ô 6 trước (gặp khi duyệt ngược ở `i = 2`), `12` vào ô 5 sau (ở `i = 5`). Vì duyệt ngược gặp `14` trước `12`, và điền ngược, nên thứ tự cuối cùng là `12, 14` — đúng thứ tự gốc.

### 8.2 Trace cách 5 (chia để trị + xoay) trên một mảng tự dựng (đã đối chiếu cách 3 — đối chiếu lại đề gốc)

In [13]:
def trace_stable_partition(a, l, r, pred, depth=0):
    pad = "    " * depth
    if r - l <= 1:
        b = l + 1 if (r - l == 1 and pred(a[l])) else l
        return b
    m = (l + r) // 2
    b1 = trace_stable_partition(a, l, m, pred, depth + 1)
    b2 = trace_stable_partition(a, m, r, pred, depth + 1)
    before = a[l:r]
    rotate(a, b1, m, b2)
    print(f"  {pad}đoạn [{l},{r}): {before} -> xoay F1={before[b1 - l:m - l]} với T2={before[m - l:b2 - l]} -> {a[l:r]}")
    return b1 + (b2 - m)

demo = [7, 1, 9, 4, 4, 2, 8, 3]
p = 4
a = list(demo)
b = trace_stable_partition(a, 0, len(a), lambda x: x < p)
print("  sau lượt 1 (x < 4 lên trước):", a, "ranh giới", b)
trace_stable_partition(a, b, len(a), lambda x: x == p)
print("  sau lượt 2 (x == 4 lên trước phần còn lại):", a)
print("  cách 3:", pivot_three_lists(demo, p))

          đoạn [0,2): [7, 1] -> xoay F1=[7] với T2=[1] -> [1, 7]
          đoạn [2,4): [9, 4] -> xoay F1=[9] với T2=[] -> [9, 4]
      đoạn [0,4): [1, 7, 9, 4] -> xoay F1=[7] với T2=[] -> [1, 7, 9, 4]
          đoạn [4,6): [4, 2] -> xoay F1=[4] với T2=[2] -> [2, 4]
          đoạn [6,8): [8, 3] -> xoay F1=[8] với T2=[3] -> [3, 8]
      đoạn [4,8): [2, 4, 3, 8] -> xoay F1=[4] với T2=[3] -> [2, 3, 4, 8]
  đoạn [0,8): [1, 7, 9, 4, 2, 3, 4, 8] -> xoay F1=[7, 9, 4] với T2=[2, 3] -> [1, 2, 3, 7, 9, 4, 4, 8]
  sau lượt 1 (x < 4 lên trước): [1, 2, 3, 7, 9, 4, 4, 8] ranh giới 3
      đoạn [3,5): [7, 9] -> xoay F1=[7] với T2=[] -> [7, 9]
          đoạn [6,8): [4, 8] -> xoay F1=[] với T2=[] -> [4, 8]
      đoạn [5,8): [4, 4, 8] -> xoay F1=[] với T2=[4] -> [4, 4, 8]
  đoạn [3,8): [7, 9, 4, 4, 8] -> xoay F1=[7, 9] với T2=[4, 4] -> [4, 4, 7, 9, 8]
  sau lượt 2 (x == 4 lên trước phần còn lại): [1, 2, 3, 4, 4, 7, 9, 8]
  cách 3: [1, 2, 3, 4, 4, 7, 9, 8]


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def pivotArray(self, nums: List[int], pivot: int) -> List[int]:
        n = len(nums)
        res = [pivot] * n
        left, right = 0, n - 1
        for i in range(n):
            if nums[i] < pivot:              # nhóm nhỏ: duyệt xuôi, điền xuôi
                res[left] = nums[i]
                left += 1
            j = n - 1 - i
            if nums[j] > pivot:              # nhóm lớn: duyệt ngược, điền ngược
                res[right] = nums[j]
                right -= 1
        return res

for nums_, p_, exp in examples:
    assert Solution().pivotArray(nums_, p_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Tính duy nhất của đáp án.** Điều kiện 1–2 cố định **tập** giá trị ở mỗi đoạn `[0, L)`, `[L, L+E)`, `[L+E, n)`; điều kiện 3 cố định **thứ tự** trong đoạn đầu và đoạn cuối; đoạn giữa toàn giá trị `pivot`. ⇒ chỉ có một mảng thỏa mãn. Vì vậy ta có thể kiểm thử bằng **so sánh bằng** với một cách đúng hiển nhiên (cách 3).

**Cách 3.** `less` nhận các phần tử nhỏ hơn đúng theo thứ tự duyệt (thứ tự gốc); tương tự `greater`; giữa là `E` bản sao `pivot`. Đúng định nghĩa.

**Cách 4.** Bất biến sau bước `i`:
- `res[0:left]` = các phần tử `< pivot` trong `nums[0..i]`, đúng thứ tự gốc;
- `res[right+1:n]` = các phần tử `> pivot` trong `nums[n−1−i..n−1]`, đúng thứ tự gốc (vì phần tử gặp sau khi duyệt ngược nằm **trước** trong mảng gốc và được ghi vào ô **bên trái** hơn).

Sau bước `n − 1`: `left = L`, `right = n − 1 − G`, đoạn giữa dài `n − L − G = E` đã sẵn giá trị `pivot`. Các vùng ghi không chồng nhau (mục 7.3).

**Cách 5.** Quy nạp theo độ dài: hai nửa đã được phân hoạch ổn định thành `T₁F₁` và `T₂F₂`. Phép xoay chỉ hoán đổi **khối** `F₁` và `T₂`, giữ thứ tự bên trong mỗi khối ⇒ `T₁T₂F₁F₂`: mọi phần tử đúng đứng trước mọi phần tử sai, và trong mỗi loại thứ tự gốc được giữ (phần tử nửa trái trước nửa phải). Hai lượt vị từ `x < p` rồi `x == p` cho đúng ba nhóm.

**Cách 1.** Mỗi lần đổi làm giảm số cặp nghịch thế theo nhãn đi 1 và không bao giờ đổi hai phần tử cùng nhãn ⇒ dừng ở trạng thái nhãn không giảm, ổn định.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Bubble theo nhãn | `O(n²)` | `O(n)` (bản sao) |
| 2. `sorted` theo nhãn | `O(n log n)` | `O(n)` |
| 3. Ba danh sách | `O(n)` | `O(n)` |
| 4. Hai con trỏ ngược chiều | `O(n)` | `O(1)` ngoài kết quả |
| 5. Tại chỗ chia để trị + xoay | `O(n log n)` | `O(log n)` |

In [15]:
rnd = random.Random(2161)
big = [rnd.randint(-10**6, 10**6) for _ in range(10**5)]
pv = big[rnd.randrange(len(big))]
for name, fn in [("sorted", pivot_sorted), ("three_lists", pivot_three_lists), ("two_pointers", pivot_two_pointers),
                 ("inplace", pivot_inplace), ("Solution", Solution().pivotArray)]:
    t0 = time.perf_counter(); r = fn(big, pv); dt = time.perf_counter() - t0
    print(f"{name:<13}: {dt * 1000:8.1f} ms")
small = big[:2000]
t0 = time.perf_counter(); pivot_bubble(small, small[0]); print(f"bubble (n=2000): {(time.perf_counter() - t0) * 1000:8.1f} ms")

sorted       :      6.1 ms
three_lists  :      3.6 ms
two_pointers :     10.6 ms
inplace      :     82.1 ms
Solution     :      8.3 ms


bubble (n=2000):     33.5 ms


## 12. Bộ test

In [16]:
def is_valid(nums, pivot, out):
    """Validator độc lập theo 3 điều kiện của đề (không dùng lời giải nào)."""
    if sorted(out) != sorted(nums):
        return False
    labels = [(x > pivot) - (x < pivot) for x in out]
    if labels != sorted(labels):                       # < trước, = giữa, > sau
        return False
    less_in = [x for x in nums if x < pivot]; greater_in = [x for x in nums if x > pivot]
    return [x for x in out if x < pivot] == less_in and [x for x in out if x > pivot] == greater_in

ALL = {
    "bubble": pivot_bubble,
    "sorted": pivot_sorted,
    "three_lists": pivot_three_lists,
    "two_pointers": pivot_two_pointers,
    "inplace": pivot_inplace,
    "Solution": lambda a, p: Solution().pivotArray(a, p),
}

def check(nums, pivot, exp=None, skip=()):
    snapshot = list(nums)
    ref = pivot_three_lists(nums, pivot)
    assert is_valid(nums, pivot, ref)
    for name, fn in ALL.items():
        if name in skip:
            continue
        got = fn(nums, pivot)
        assert got == ref, (name, nums, pivot, got, ref)
        assert nums == snapshot, name + " đã sửa đầu vào"
    if exp is not None:
        assert ref == exp
    return ref

for nums_, p_, exp in examples:
    check(nums_, p_, exp)
print("Ví dụ đề: OK")

check([5], 5, [5])                                    # một phần tử
check([2, 2, 2], 2, [2, 2, 2])                        # toàn pivot
check([1, 2, 3], 3, [1, 2, 3])                        # pivot lớn nhất
check([3, 2, 1], 1, [1, 3, 2])                        # pivot nhỏ nhất: nhóm lớn giữ thứ tự 3, 2
check([4, 1, 4, 1, 4], 4, [1, 1, 4, 4, 4])
check([-10**6, 10**6, 0], 0, [-10**6, 0, 10**6])
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(3000):
    n = rnd.randint(1, 12)
    a = [rnd.randint(-4, 4) for _ in range(n)]
    check(a, rnd.choice(a))
print("3000 mảng nhỏ (nhiều giá trị trùng) đối chiếu 6 cách + validator: OK")

for _ in range(10):
    n = rnd.randint(1, 10**5)
    a = [rnd.randint(-10**6, 10**6) for _ in range(n)]
    check(a, rnd.choice(a), skip=("bubble",))
print("10 mảng lớn (bỏ bubble O(n²)): OK")

Ví dụ đề: OK
Edge case: OK


3000 mảng nhỏ (nhiều giá trị trùng) đối chiếu 6 cách + validator: OK


10 mảng lớn (bỏ bubble O(n²)): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng phân hoạch kiểu Quicksort (Lomuto), không ổn định

In [17]:
def pivot_bug_lomuto(nums, pivot):
    a = list(nums)
    i = 0
    for j in range(len(a)):                  # đưa phần tử < pivot lên đầu bằng swap
        if a[j] < pivot:
            a[i], a[j] = a[j], a[i]; i += 1
    k = i
    for j in range(i, len(a)):               # đưa phần tử == pivot lên tiếp theo
        if a[j] == pivot:
            a[k], a[j] = a[j], a[k]; k += 1
    return a

for nums_, p_ in [([9, 12, 5, 10, 14, 3, 10], 10), ([3, 2, 1], 1), ([7, 1, 9, 4, 4, 2, 8, 3], 4)]:
    print(f"{nums_}, pivot={p_}: Lomuto = {pivot_bug_lomuto(nums_, p_)} | đúng = {Solution().pivotArray(nums_, p_)}")

[9, 12, 5, 10, 14, 3, 10], pivot=10: Lomuto = [9, 5, 3, 10, 10, 12, 14] | đúng = [9, 5, 3, 10, 10, 12, 14]
[3, 2, 1], pivot=1: Lomuto = [1, 2, 3] | đúng = [1, 3, 2]
[7, 1, 9, 4, 4, 2, 8, 3], pivot=4: Lomuto = [1, 2, 3, 4, 4, 7, 8, 9] | đúng = [1, 2, 3, 4, 4, 7, 9, 8]


**Vì sao sai:** ở ví dụ 1 Lomuto **tình cờ** cho đúng — rất nguy hiểm vì test mẫu qua! Nhưng với `[3, 2, 1]`, `pivot = 1`, phép `swap` ném `3` (phần tử đầu) ra cuối ⇒ nhóm lớn thành `[2, 3]` thay vì `[3, 2]`. Mảng thứ ba cũng bị đảo thứ tự nhóm lớn. Phép đổi chỗ "nhảy xa" luôn có nguy cơ cho một phần tử vượt qua phần tử cùng nhóm. **Sửa:** dùng cách 3/4 (ghi sang mảng mới) hoặc cách 5 (xoay khối, không swap từng phần tử).

### Lỗi 2 — Duyệt xuôi nhưng điền nhóm lớn từ phải sang

In [18]:
def pivot_bug_fill_direction(nums, pivot):
    n = len(nums); res = [pivot] * n
    left, right = 0, n - 1
    for x in nums:                            # SAI: nhóm lớn cũng duyệt xuôi nhưng điền ngược
        if x < pivot:
            res[left] = x; left += 1
        elif x > pivot:
            res[right] = x; right -= 1
    return res

nums_, p_ = [-3, 4, 3, 2], 2
print("sai :", pivot_bug_fill_direction(nums_, p_), "| đúng:", Solution().pivotArray(nums_, p_))

sai : [-3, 2, 3, 4] | đúng: [-3, 2, 4, 3]


**Vì sao sai:** duyệt xuôi + điền ngược ⇒ **đảo ngược** nhóm lớn (`[3, 4]` thay vì `[4, 3]`). **Sửa:** nhóm lớn phải duyệt **ngược** khi điền ngược (cách 4), hoặc đếm trước `L + E` rồi điền xuôi từ vị trí đó.

### Lỗi 3 — Giả sử `pivot` chỉ xuất hiện một lần

In [19]:
def pivot_bug_single(nums, pivot):
    return [x for x in nums if x < pivot] + [pivot] + [x for x in nums if x > pivot]

nums_, p_ = [9, 12, 5, 10, 14, 3, 10], 10
out = pivot_bug_single(nums_, p_)
print("sai :", out, "(độ dài", len(out), ") | đúng:", Solution().pivotArray(nums_, p_))

sai : [9, 5, 3, 10, 12, 14] (độ dài 6 ) | đúng: [9, 5, 3, 10, 10, 12, 14]


**Sửa:** `[pivot] * nums.count(pivot)` hoặc đếm trong lúc duyệt.

### Lỗi 4 — Gộp "bằng" vào nhóm trái bằng `<=`

In [20]:
def pivot_bug_le(nums, pivot):
    return [x for x in nums if x <= pivot] + [x for x in nums if x > pivot]

nums_, p_ = [10, 5, 12], 10
print("sai :", pivot_bug_le(nums_, p_), "| đúng:", Solution().pivotArray(nums_, p_))

sai : [10, 5, 12] | đúng: [5, 10, 12]


**Vì sao sai:** các `pivot` bị giữ ở vị trí gốc **lẫn trong** nhóm nhỏ, không nằm ở giữa. Với ví dụ 1 lỗi bị che khuất (các `10` tình cờ đứng sau mọi số nhỏ) — phải test trường hợp `pivot` đứng **trước** một số nhỏ hơn.

### Lỗi 5 — Gọi `sorted(nums)` (sắp theo giá trị)

In [21]:
def pivot_bug_sort_values(nums, pivot):
    return sorted(nums)                       # SAI: sắp lại thứ tự bên trong mỗi nhóm

nums_, p_ = [9, 12, 5, 10, 14, 3, 10], 10
print("sai :", pivot_bug_sort_values(nums_, p_), "| đúng:", Solution().pivotArray(nums_, p_))

sai : [3, 5, 9, 10, 10, 12, 14] | đúng: [9, 5, 3, 10, 10, 12, 14]


**Vì sao sai:** điều kiện 1–2 thỏa, nhưng điều kiện 3 (giữ thứ tự tương đối) bị phá: `[3, 5, 9]` thay vì `[9, 5, 3]`. **Sửa:** sắp theo **nhãn nhóm**, không theo giá trị (cách 2).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Giữ thứ tự tương đối" ⇒ **ổn định** ⇒ không dùng swap kiểu Quicksort; ghi sang mảng mới.
- Chỉ vài nhóm ⇒ phân phối kiểu counting sort `O(n)`.
- Python một dòng: `sorted(nums, key=lambda x: (x > p) - (x < p))` (Timsort ổn định).
- Hai con trỏ ngược chiều: **duyệt ngược + điền ngược** = giữ thứ tự xuôi.
- Phần tử giống hệt nhau (bằng `pivot`) chỉ cần **đếm**, không cần lưu.
- Test phải có trường hợp nhóm lớn có ≥ 2 phần tử **không tăng** (để bắt lỗi đảo thứ tự) và `pivot` xuất hiện nhiều lần.

## 15. Biến thể của bài

1. **LeetCode 75 — Sort Colors:** ba giá trị 0/1/2, tại chỗ một lượt (Dutch National Flag) — **không cần** ổn định vì các phần tử cùng giá trị giống hệt nhau.
2. **LeetCode 905 — Sort Array By Parity / 922:** phân hoạch theo chẵn/lẻ (bản ổn định: hai danh sách).
3. **LeetCode 86 — Partition List:** cùng ý tưởng trên **danh sách liên kết** (hai danh sách giả rồi nối) — tự nhiên ổn định và `O(1)` bộ nhớ.
4. **Phân hoạch theo khoảng `[lo, hi]`** (ba nhóm `< lo`, trong khoảng, `> hi`) — bài tập 3.
5. **Phân hoạch ổn định theo `k` nhóm** (nhãn `0..k−1`): counting sort ổn định — bài tập 5.

In [22]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val; self.next = next

def build_list(vals):
    dummy = ListNode(); cur = dummy
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

def to_list(head):
    out = []
    while head:
        out.append(head.val); head = head.next
    return out

def partition_linked(head, pivot):
    """Biến thể 3 nhóm trên danh sách liên kết: ba đuôi giả, nối lại — O(1) bộ nhớ phụ."""
    heads = [ListNode(), ListNode(), ListNode()]
    tails = heads[:]
    while head:
        nxt = head.next; head.next = None
        g = (head.val > pivot) - (head.val < pivot) + 1
        tails[g].next = head; tails[g] = head
        head = nxt
    tails[0].next = heads[1].next or heads[2].next
    tails[1].next = heads[2].next
    return heads[0].next or heads[1].next or heads[2].next

print("danh sách liên kết:", to_list(partition_linked(build_list([9, 12, 5, 10, 14, 3, 10]), 10)))
rnd_v = random.Random(15)
for _ in range(500):
    a = [rnd_v.randint(-3, 3) for _ in range(rnd_v.randint(1, 10))]
    p = rnd_v.choice(a)
    assert to_list(partition_linked(build_list(a), p)) == pivot_three_lists(a, p)
print("biến thể danh sách liên kết: OK")

danh sách liên kết: [9, 5, 3, 10, 10, 12, 14]
biến thể danh sách liên kết: OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Bubble theo nhãn | đổi kề khi nhãn giảm | `O(n²)` | `O(n)` | minh họa ổn định |
| 2. `sorted` theo nhãn | Timsort ổn định | `O(n log n)` | `O(n)` | một dòng, phỏng vấn nhanh |
| 3. Ba danh sách | phân phối | `O(n)` | `O(n)` | dễ hiểu, dễ chứng minh |
| 4. Hai con trỏ ngược chiều | duyệt ngược + điền ngược | `O(n)` | `O(1)` + kết quả | **nộp bài** |
| 5. Chia để trị + xoay | stable_partition tại chỗ | `O(n log n)` | `O(log n)` | khi không có bộ nhớ phụ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().pivotArray([5, 1, 5, 7, 2, 9, 5, 0], 5)`?

**Bài 2 (tính tay):** Với `nums = [8, 3, 6, 1, 6, 9]`, `pivot = 6`, chạy cách 4 bằng tay: sau mỗi `i`, `left`, `right` bằng bao nhiêu?

**Bài 3 (điền TODO):** Hoàn thiện `partition_range(nums, lo, hi)` — ba nhóm `< lo`, `lo ≤ x ≤ hi`, `> hi`, **cả ba nhóm** đều giữ thứ tự gốc.

**Bài 4 (tự viết lại):** Viết lại cách 4 dùng **hai vòng lặp riêng**: vòng 1 đếm `L`, `E`; vòng 2 điền xuôi ba con trỏ `p0 = 0`, `p1 = L`, `p2 = L + E`.

**Bài 5 (mini challenge):** Viết `stable_group(nums, key, k)` phân hoạch ổn định theo nhãn `key(x) ∈ {0, …, k−1}` trong `O(n + k)` (counting sort ổn định: đếm, cộng dồn, điền). Kiểm bằng `sorted(nums, key=key)`.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def partition_range(nums, lo, hi):
    # TODO: ba danh sách theo điều kiện x < lo, lo <= x <= hi, x > hi
    pass

# Bài 4 — TODO
def pivot_three_pointers(nums, pivot):
    # TODO: đếm L, E; rồi p0, p1, p2
    pass

# Bài 5 — TODO
def stable_group(nums, key, k):
    # TODO: cnt[key], cộng dồn thành vị trí bắt đầu, điền xuôi
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Nhỏ hơn 5: `1, 2, 0` (theo thứ tự gốc).
- Mức 2: Có ba số 5; lớn hơn 5: `7, 9`.
- Mức 3: `[1, 2, 0, 5, 5, 5, 7, 9]`.

**Bài 2**
- Mức 1: Duyệt xuôi `8,3,6,1,6,9`, duyệt ngược `9,6,1,6,3,8`.
- Mức 2: `left` tăng ở `i = 1, 3`; `right` giảm ở `i = 0` (gặp 9) và `i = 5` (gặp 8).
- Mức 3: Kết quả `[3, 1, 6, 6, 8, 9]`.

**Bài 3**
- Mức 1: Giống cách 3, đổi điều kiện.
- Mức 2: Nhóm giữa **không** giống hệt nhau ⇒ phải lưu cả danh sách.
- Mức 3: Kiểm bằng `sorted(nums, key=nhãn)`.

**Bài 4**
- Mức 1: `L = sum(x < p)`, `E = sum(x == p)`.
- Mức 2: `p2` bắt đầu ở `L + E`.
- Mức 3: Mỗi phần tử ghi vào con trỏ nhóm của nó rồi tăng con trỏ.

**Bài 5**
- Mức 1: `start[g] = tổng cnt[0..g−1]`.
- Mức 2: Duyệt xuôi, `res[start[g]] = x; start[g] += 1`.
- Mức 3: Đây là bước phân phối của counting sort / radix sort.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1
r1 = Solution().pivotArray([5, 1, 5, 7, 2, 9, 5, 0], 5)
print("Bài 1:", r1); assert r1 == [1, 2, 0, 5, 5, 5, 7, 9]

# Bài 2
print("Bài 2:"); r2 = trace_two_pointers([8, 3, 6, 1, 6, 9], 6); assert r2 == [3, 1, 6, 6, 8, 9]

# Bài 3
def partition_range_sol(nums, lo, hi):
    return [x for x in nums if x < lo] + [x for x in nums if lo <= x <= hi] + [x for x in nums if x > hi]

rnd = random.Random(3)
for _ in range(1000):
    a = [rnd.randint(-5, 5) for _ in range(rnd.randint(1, 12))]
    lo = rnd.randint(-5, 5); hi = rnd.randint(lo, 5)
    assert partition_range_sol(a, lo, hi) == sorted(a, key=lambda x: 0 if x < lo else (1 if x <= hi else 2))
print("Bài 3 OK, ví dụ:", partition_range_sol([7, 1, 4, 9, 3, 5, 0], 3, 5))

Bài 1: [1, 2, 0, 5, 5, 5, 7, 9]
Bài 2:
   i | nums[i] nums[n-1-i] | left right | res
   0 |       8           9 |    0     4 | ['_', '_', '_', '_', '_', 9]
   1 |       3           6 |    1     4 | [3, '_', '_', '_', '_', 9]
   2 |       6           1 |    1     4 | [3, '_', '_', '_', '_', 9]
   3 |       1           6 |    2     4 | [3, 1, '_', '_', '_', 9]
   4 |       6           3 |    2     4 | [3, 1, '_', '_', '_', 9]
   5 |       9           8 |    2     3 | [3, 1, '_', '_', 8, 9]
  điền pivot vào đoạn [2, 3] -> [3, 1, 6, 6, 8, 9]
Bài 3 OK, ví dụ: [1, 0, 4, 3, 5, 7, 9]


In [25]:
# Bài 4
def pivot_three_pointers_sol(nums, pivot):
    L = sum(1 for x in nums if x < pivot); E = sum(1 for x in nums if x == pivot)
    res = [0] * len(nums)
    ptr = [0, L, L + E]
    for x in nums:
        g = (x > pivot) - (x < pivot) + 1
        res[ptr[g]] = x; ptr[g] += 1
    return res

# Bài 5
def stable_group_sol(nums, key, k):
    cnt = [0] * k
    for x in nums:
        cnt[key(x)] += 1
    start = [0] * k
    for g in range(1, k):
        start[g] = start[g - 1] + cnt[g - 1]
    res = [None] * len(nums)
    for x in nums:
        g = key(x); res[start[g]] = x; start[g] += 1
    return res

rnd = random.Random(5)
for _ in range(1500):
    a = [rnd.randint(-6, 6) for _ in range(rnd.randint(1, 15))]
    p = rnd.choice(a)
    assert pivot_three_pointers_sol(a, p) == Solution().pivotArray(a, p)
    k = rnd.randint(1, 5)
    key = lambda x, k=k: x % k
    assert stable_group_sol(a, key, k) == sorted(a, key=key)
print("Bài 4, Bài 5 OK; ví dụ nhóm theo x % 3:", stable_group_sol([5, 3, 4, 9, 1, 6, 2], lambda x: x % 3, 3))

Bài 4, Bài 5 OK; ví dụ nhóm theo x % 3: [3, 9, 6, 4, 1, 5, 2]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template phân hoạch ổn định 3 nhóm, `O(n)`:**

```python
res = [pivot] * n
left, right = 0, n - 1
for i in range(n):
    if nums[i] < pivot: res[left] = nums[i]; left += 1           # xuôi - xuôi
    if nums[n-1-i] > pivot: res[right] = nums[n-1-i]; right -= 1 # ngược - ngược
```

**Một dòng:** `sorted(nums, key=lambda x: (x > pivot) - (x < pivot))`.

**Checklist:**
- [ ] Không dùng swap kiểu Quicksort (mất ổn định)?
- [ ] Nhóm lớn: duyệt và điền **cùng chiều** (cả hai xuôi hoặc cả hai ngược)?
- [ ] `pivot` xuất hiện nhiều lần được giữ đủ số lượng?
- [ ] Phần tử bằng `pivot` nằm **giữa**, không lẫn vào nhóm nhỏ?
- [ ] Test: nhóm lớn không tăng, `pivot` nhỏ nhất / lớn nhất, toàn `pivot`?
- [ ] Chữ ký đúng: `def pivotArray(self, nums: List[int], pivot: int) -> List[int]`?